In [ ]:
# ============================================================
# BACKTEST — Solo Long Selectivo
# Entra cuando predice SUBE, fuera cuando predice BAJA o INCIERTO
# ============================================================

LOWER = 0.46
UPPER = 0.54

# --- Reconstruir test set con predicciones ---
df_ml_t5 = df_t5.dropna(subset=['ret_t5']).copy()
df_ml_t5 = df_ml_t5.sort_values('Date').reset_index(drop=True)
split_idx = int(len(df_ml_t5) * 0.8)
df_test_full = df_ml_t5.iloc[split_idx:].copy()
df_test_full['Date']       = pd.to_datetime(df_test_full['Date'])
df_test_full['proba']      = proba_t5
df_test_full['pred']       = np.where(proba_t5 > UPPER, 1,
                             np.where(proba_t5 < LOWER, 0, 2))
df_test_full['opera_long'] = df_test_full['pred'] == 1  # solo longs
df_test_full['ret_estrat'] = np.where(df_test_full['opera_long'],
                                       df_test_full['ret_t5'], 0)

# --- Cargar precios para B&H ---
df_precios = pd.read_csv(r"C:\Users\Ricardo\TFG_ENTREGAR\data\mid\df_merged_news_return.csv",
                          usecols=['Date', 'Ticker', 'Price Close'])
df_precios['Date'] = pd.to_datetime(df_precios['Date'])
df_precios = df_precios.dropna(subset=['Price Close'])

# --- Backtest por banco ---
resultados_banco = []
coste_por_op     = 0.004

for ticker, grupo in df_test_full.groupby('Ticker'):
    grupo = grupo.sort_values('Date').reset_index(drop=True)

    ops   = grupo[grupo['opera_long']].copy()
    n_ops = len(ops)
    if n_ops == 0:
        continue

    n_ac  = ((ops['ret_t5'] > 0)).sum()
    hit   = n_ac / n_ops

    ret_acum_strat      = (1 + ops['ret_t5']).prod() - 1
    ret_acum_strat_neto = (1 + ops['ret_t5'] - coste_por_op).prod() - 1

    # B&H con precios reales
    precios_ticker = df_precios[df_precios['Ticker'] == ticker].sort_values('Date')
    fecha_ini      = grupo['Date'].min()
    fecha_fin      = grupo['Date'].max()
    precios_rango  = precios_ticker[
        (precios_ticker['Date'] >= fecha_ini) &
        (precios_ticker['Date'] <= fecha_fin)
    ]
    ret_acum_bh = (precios_rango['Price Close'].iloc[-1] /
                   precios_rango['Price Close'].iloc[0]) - 1 \
                   if len(precios_rango) >= 2 else np.nan

    resultados_banco.append({
        'Ticker'          : ticker,
        'N_ops'           : n_ops,
        'Hit Rate'        : hit,
        'Ret_medio_op'    : ops['ret_t5'].mean(),
        'Acum_bruto'      : ret_acum_strat,
        'Acum_neto'       : ret_acum_strat_neto,
        'Acum_BH'         : ret_acum_bh,
        'Vence_BH_bruto'  : ret_acum_strat > ret_acum_bh,
        'Vence_BH_neto'   : ret_acum_strat_neto > ret_acum_bh
    })

df_result = pd.DataFrame(resultados_banco).sort_values('Acum_bruto', ascending=False)

# --- Print ---
print("=" * 95)
print("  BACKTEST SOLO LONG — Por Banco")
print(f"  Umbrales: Incierto [{LOWER}, {UPPER}]  |  Sube > {UPPER}  |  Costes: {coste_por_op*100:.1f}% por vuelta")
print("=" * 95)
print(f"  {'Ticker':<12} {'Ops':>5}  {'Hit Rate':>9}  {'Ret/Op':>8}  "
      f"{'Acum Bruto':>11}  {'Acum Neto':>10}  {'B&H':>8}  {'Bruto':>6}  {'Neto':>6}")
print("-" * 95)
for _, r in df_result.iterrows():
    bh   = f"{r['Acum_BH']*100:.1f}%"   if not np.isnan(r['Acum_BH']) else "N/A"
    mb   = "✅" if r['Vence_BH_bruto'] else "❌"
    mn   = "✅" if r['Vence_BH_neto']  else "❌"
    print(f"  {r['Ticker']:<12} {int(r['N_ops']):>5}  {r['Hit Rate']:>9.1%}  "
          f"{r['Ret_medio_op']:>8.2%}  "
          f"{r['Acum_bruto']:>11.1%}  "
          f"{r['Acum_neto']:>10.1%}  "
          f"{bh:>8}  {mb:>6}  {mn:>6}")
print("-" * 95)
print(f"\n  Bancos que vencen al B&H bruto : {df_result['Vence_BH_bruto'].sum()}/{len(df_result)}")
print(f"  Bancos que vencen al B&H neto  : {df_result['Vence_BH_neto'].sum()}/{len(df_result)}")
print("=" * 95)

# --- Tests estadísticos globales ---
from scipy import stats
from scipy.stats import binomtest

ops_global = df_test_full[df_test_full['opera_long']].copy()
retornos   = ops_global['ret_t5'].values
n_total    = len(retornos)
n_ac       = (retornos > 0).sum()

binom_p = binomtest(n_ac, n_total, p=0.5, alternative='greater').pvalue
t_stat, t_p = stats.ttest_1samp(retornos, popmean=0, alternative='greater')
ic = stats.t.interval(0.95, df=n_total-1,
                       loc=retornos.mean(), scale=stats.sem(retornos))

print(f"\n{'='*60}")
print(f"  TESTS ESTADÍSTICOS GLOBALES — {n_total} operaciones")
print(f"{'='*60}")
print(f"  Hit Rate            : {n_ac/n_total:.1%}  (p-binomial: {binom_p:.4f})")
print(f"  Retorno medio/op    : {retornos.mean()*100:.2f}%")
print(f"  p-valor test t      : {t_p:.4f}")
print(f"  IC 95%              : [{ic[0]*100:.2f}%, {ic[1]*100:.2f}%]")
print(f"  {'✅ Significativo al 95%' if t_p < 0.05 else '❌ No significativo'}")
print(f"{'='*60}")

# --- Gráfico ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

x     = range(len(df_result))
width = 0.28

axes[0].bar([i - width for i in x], df_result['Acum_bruto'] * 100,
            width=width, label='Estrategia bruta', color='steelblue', alpha=0.8)
axes[0].bar([i for i in x],         df_result['Acum_neto'] * 100,
            width=width, label='Estrategia neta',  color='teal',      alpha=0.8)
axes[0].bar([i + width for i in x], df_result['Acum_BH'] * 100,
            width=width, label='Buy & Hold',        color='salmon',    alpha=0.8)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(df_result['Ticker'], rotation=45)
axes[0].set_title('Retorno Acumulado por Banco')
axes[0].set_ylabel('Retorno (%)')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(df_result['Ticker'], df_result['Hit Rate'] * 100,
            color='steelblue', alpha=0.8)
axes[1].axhline(50, color='red',    linestyle='--', linewidth=1.5, label='50% azar')
axes[1].axhline(df_result['Hit Rate'].mean() * 100, color='orange',
                linestyle='--', linewidth=1.5,
                label=f"Media {df_result['Hit Rate'].mean():.1%}")
axes[1].set_title('Hit Rate por Banco')
axes[1].set_ylabel('Hit Rate (%)')
axes[1].set_xticklabels(df_result['Ticker'], rotation=45)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

axes[2].bar(df_result['Ticker'], df_result['N_ops'],
            color='steelblue', alpha=0.8)
axes[2].set_title('Nº Operaciones por Banco')
axes[2].set_ylabel('Operaciones')
axes[2].set_xticklabels(df_result['Ticker'], rotation=45)
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Backtest Solo Long — Umbrales [{LOWER}, {UPPER}]',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# DCA SELECTIVO — Compras 1 unidad cada vez que el modelo dice SUBE
# Nunca vendes. Al final comparas precio medio vs precio final.
# ============================================================

LOWER = 0.46
UPPER = 0.54

# --- Reconstruir test set ---
df_ml_t5 = df_t5.dropna(subset=['ret_t5']).copy()
df_ml_t5 = df_ml_t5.sort_values('Date').reset_index(drop=True)
split_idx = int(len(df_ml_t5) * 0.8)
df_test_full = df_ml_t5.iloc[split_idx:].copy()
df_test_full['Date']  = pd.to_datetime(df_test_full['Date'])
df_test_full['proba'] = proba_t5
df_test_full['pred']  = np.where(proba_t5 > UPPER, 1,
                        np.where(proba_t5 < LOWER, 0, 2))

# --- Cargar precios ---
df_precios = pd.read_csv(r"C:\Users\Ricardo\TFG_ENTREGAR\data\mid\df_merged_news_return.csv",
                          usecols=['Date', 'Ticker', 'Price Close'])
df_precios['Date'] = pd.to_datetime(df_precios['Date'])
df_precios = df_precios.dropna(subset=['Price Close'])

resultados = []

for ticker, grupo in df_test_full.groupby('Ticker'):
    grupo = grupo.sort_values('Date').reset_index(drop=True)

    # Precios diarios del período de test
    precios_ticker = df_precios[df_precios['Ticker'] == ticker].sort_values('Date')
    fecha_ini      = grupo['Date'].min()
    fecha_fin      = grupo['Date'].max()
    precios_rango  = precios_ticker[
        (precios_ticker['Date'] >= fecha_ini) &
        (precios_ticker['Date'] <= fecha_fin)
    ].reset_index(drop=True)

    if len(precios_rango) < 2:
        continue

    precio_final = precios_rango['Price Close'].iloc[-1]

    # --- DCA Selectivo: compras cuando modelo dice SUBE ---
    compras_dca = []
    for _, fila in grupo[grupo['pred'] == 1].iterrows():
        # Precio más cercano a la fecha de señal
        idx = (precios_rango['Date'] - fila['Date']).abs().idxmin()
        precio_compra = precios_rango['Price Close'].iloc[idx]
        compras_dca.append(precio_compra)

    # --- DCA Puro: compras en cada observación sin filtro ---
    compras_bh = []
    for _, fila in grupo.iterrows():
        idx = (precios_rango['Date'] - fila['Date']).abs().idxmin()
        precio_compra = precios_rango['Price Close'].iloc[idx]
        compras_bh.append(precio_compra)

    if len(compras_dca) == 0:
        continue

    precio_medio_dca = np.mean(compras_dca)
    precio_medio_bh  = np.mean(compras_bh)
    n_compras_dca    = len(compras_dca)
    n_compras_bh     = len(compras_bh)

    # Retorno = (precio_final - precio_medio) / precio_medio
    ret_dca = (precio_final - precio_medio_dca) / precio_medio_dca
    ret_bh  = (precio_final - precio_medio_bh)  / precio_medio_bh

    resultados.append({
        'Ticker'          : ticker,
        'N_compras_DCA'   : n_compras_dca,
        'N_compras_BH'    : n_compras_bh,
        'Precio_medio_DCA': precio_medio_dca,
        'Precio_medio_BH' : precio_medio_bh,
        'Precio_final'    : precio_final,
        'Ret_DCA'         : ret_dca,
        'Ret_BH'          : ret_bh,
        'Vence_BH'        : ret_dca > ret_bh
    })

df_dca = pd.DataFrame(resultados).sort_values('Ret_DCA', ascending=False)

# --- Print ---
print("=" * 95)
print("  DCA SELECTIVO — Compras 1 unidad cuando modelo dice SUBE, nunca vendes")
print(f"  Umbrales: [{LOWER}, {UPPER}]")
print("=" * 95)
print(f"  {'Ticker':<12} {'Compras':>8}  {'P.Medio DCA':>12}  {'P.Medio BH':>11}  "
      f"{'P.Final':>9}  {'Ret DCA':>9}  {'Ret BH':>9}  {'Gana':>6}")
print("-" * 95)
for _, r in df_dca.iterrows():
    marca = "✅" if r['Vence_BH'] else "❌"
    print(f"  {r['Ticker']:<12} {int(r['N_compras_DCA']):>8}  "
          f"{r['Precio_medio_DCA']:>12.2f}  "
          f"{r['Precio_medio_BH']:>11.2f}  "
          f"{r['Precio_final']:>9.2f}  "
          f"{r['Ret_DCA']:>9.1%}  "
          f"{r['Ret_BH']:>9.1%}  "
          f"{marca:>6}")
print("-" * 95)
print(f"\n  Bancos que vencen al DCA puro : {df_dca['Vence_BH'].sum()}/{len(df_dca)}")
print("=" * 95)

# --- Gráfico ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x     = range(len(df_dca))
width = 0.35

axes[0].bar([i - width/2 for i in x], df_dca['Ret_DCA'] * 100,
            width=width, label='DCA Selectivo (modelo)', color='steelblue', alpha=0.8)
axes[0].bar([i + width/2 for i in x], df_dca['Ret_BH'] * 100,
            width=width, label='DCA Puro (todas las semanas)', color='salmon', alpha=0.8)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(df_dca['Ticker'], rotation=45)
axes[0].set_title('Retorno DCA Selectivo vs DCA Puro')
axes[0].set_ylabel('Retorno total (%)')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar([i - width/2 for i in x], df_dca['Precio_medio_DCA'],
            width=width, label='Precio medio DCA Selectivo', color='steelblue', alpha=0.8)
axes[1].bar([i + width/2 for i in x], df_dca['Precio_medio_BH'],
            width=width, label='Precio medio DCA Puro', color='salmon', alpha=0.8)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(df_dca['Ticker'], rotation=45)
axes[1].set_title('Precio Medio de Entrada')
axes[1].set_ylabel('Precio')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle(f'DCA Selectivo guiado por modelo — Nunca vendes',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SHARPE RATIO — DCA Selectivo vs DCA Puro (Buy & Hold)
# ============================================================

sharpe_resultados = []

for ticker, grupo in df_test_full.groupby('Ticker'):
    grupo = grupo.sort_values('Date').reset_index(drop=True)

    precios_ticker = df_precios[df_precios['Ticker'] == ticker].sort_values('Date')
    fecha_ini      = grupo['Date'].min()
    fecha_fin      = grupo['Date'].max()
    precios_rango  = precios_ticker[
        (precios_ticker['Date'] >= fecha_ini) &
        (precios_ticker['Date'] <= fecha_fin)
    ].reset_index(drop=True)

    if len(precios_rango) < 2:
        continue

    # --- Retornos diarios del activo (para Sharpe B&H) ---
    precios_rango['ret_diario'] = precios_rango['Price Close'].pct_change()
    ret_diarios_bh = precios_rango['ret_diario'].dropna().values

    # --- Retornos de la estrategia DCA selectivo ---
    # Usamos los retornos T+5 de las operaciones seleccionadas
    ops = grupo[grupo['pred'] == 1].copy()
    ret_estrategia = ops['ret_t5'].values

    if len(ret_estrategia) < 2:
        continue

    # --- Sharpe B&H (anualizado con retornos diarios) ---
    sharpe_bh = (ret_diarios_bh.mean() / ret_diarios_bh.std()) * np.sqrt(252) \
                if ret_diarios_bh.std() > 0 else 0

    # --- Sharpe Estrategia (anualizado con retornos T+5) ---
    # T+5 → ~52 operaciones posibles por año, pero solo operamos N/5 veces
    ops_por_año    = len(ret_estrategia) / 5
    sharpe_estrat  = (ret_estrategia.mean() / ret_estrategia.std()) * np.sqrt(ops_por_año) \
                     if ret_estrategia.std() > 0 else 0

    sharpe_resultados.append({
        'Ticker'        : ticker,
        'N_ops'         : len(ret_estrategia),
        'Sharpe_Estrat' : sharpe_estrat,
        'Sharpe_BH'     : sharpe_bh,
        'Gana_BH'       : sharpe_estrat > sharpe_bh
    })

df_sharpe = pd.DataFrame(sharpe_resultados).sort_values('Sharpe_Estrat', ascending=False)

# --- Print ---
print("=" * 65)
print("  SHARPE RATIO — DCA Selectivo vs Buy & Hold")
print("=" * 65)
print(f"  {'Ticker':<12} {'N_ops':>6}  {'Sharpe Estrat':>14}  {'Sharpe B&H':>11}  {'Gana':>6}")
print("-" * 65)
for _, r in df_sharpe.iterrows():
    marca = "✅" if r['Gana_BH'] else "❌"
    print(f"  {r['Ticker']:<12} {int(r['N_ops']):>6}  "
          f"{r['Sharpe_Estrat']:>14.4f}  "
          f"{r['Sharpe_BH']:>11.4f}  "
          f"{marca:>6}")
print("-" * 65)
print(f"  {'MEDIA':<12} {df_sharpe['N_ops'].mean():>6.0f}  "
      f"{df_sharpe['Sharpe_Estrat'].mean():>14.4f}  "
      f"{df_sharpe['Sharpe_BH'].mean():>11.4f}")
print("=" * 65)
print(f"\n  Bancos con mejor Sharpe que B&H: {df_sharpe['Gana_BH'].sum()}/{len(df_sharpe)}")

# --- Gráfico ---
fig, ax = plt.subplots(figsize=(12, 5))

x     = range(len(df_sharpe))
width = 0.35

ax.bar([i - width/2 for i in x], df_sharpe['Sharpe_Estrat'],
       width=width, label='DCA Selectivo', color='steelblue', alpha=0.8)
ax.bar([i + width/2 for i in x], df_sharpe['Sharpe_BH'],
       width=width, label='Buy & Hold',    color='salmon',    alpha=0.8)
ax.axhline(0, color='black',  linewidth=0.8)
ax.axhline(1, color='green',  linestyle='--', linewidth=1.2, label='Sharpe = 1 (bueno)')
ax.axhline(0.5, color='orange', linestyle='--', linewidth=1.2, label='Sharpe = 0.5 (aceptable)')
ax.set_xticks(list(x))
ax.set_xticklabels(df_sharpe['Ticker'], rotation=45)
ax.set_title('Sharpe Ratio — DCA Selectivo vs Buy & Hold', fontsize=13, fontweight='bold')
ax.set_ylabel('Sharpe Ratio')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SORTINO RATIO — DCA Selectivo vs Buy & Hold
# El Sortino solo penaliza la volatilidad NEGATIVA (downside)
# Es más justo para estrategias con retornos asimétricos
# ============================================================

sortino_resultados = []

for ticker, grupo in df_test_full.groupby('Ticker'):
    grupo = grupo.sort_values('Date').reset_index(drop=True)

    precios_ticker = df_precios[df_precios['Ticker'] == ticker].sort_values('Date')
    fecha_ini      = grupo['Date'].min()
    fecha_fin      = grupo['Date'].max()
    precios_rango  = precios_ticker[
        (precios_ticker['Date'] >= fecha_ini) &
        (precios_ticker['Date'] <= fecha_fin)
    ].reset_index(drop=True)

    if len(precios_rango) < 2:
        continue

    # --- Retornos diarios B&H ---
    precios_rango['ret_diario'] = precios_rango['Price Close'].pct_change()
    ret_diarios_bh = precios_rango['ret_diario'].dropna().values

    # --- Retornos estrategia ---
    ops            = grupo[grupo['pred'] == 1].copy()
    ret_estrategia = ops['ret_t5'].values

    if len(ret_estrategia) < 2:
        continue

    ops_por_año = len(ret_estrategia) / 5

    # --- Sortino B&H ---
    downside_bh    = ret_diarios_bh[ret_diarios_bh < 0]
    downside_std_bh = downside_bh.std() if len(downside_bh) > 1 else np.nan
    sortino_bh     = (ret_diarios_bh.mean() / downside_std_bh) * np.sqrt(252) \
                     if downside_std_bh and downside_std_bh > 0 else 0

    # --- Sortino Estrategia ---
    downside_estrat    = ret_estrategia[ret_estrategia < 0]
    downside_std_estrat = downside_estrat.std() if len(downside_estrat) > 1 else np.nan
    sortino_estrat     = (ret_estrategia.mean() / downside_std_estrat) * np.sqrt(ops_por_año) \
                         if downside_std_estrat and downside_std_estrat > 0 else 0

    sortino_resultados.append({
        'Ticker'          : ticker,
        'N_ops'           : len(ret_estrategia),
        'N_downside'      : len(downside_estrat),
        'Sortino_Estrat'  : sortino_estrat,
        'Sortino_BH'      : sortino_bh,
        'Gana_BH'         : sortino_estrat > sortino_bh
    })

df_sortino = pd.DataFrame(sortino_resultados).sort_values('Sortino_Estrat', ascending=False)

# --- Print ---
print("=" * 72)
print("  SORTINO RATIO — DCA Selectivo vs Buy & Hold")
print("  (Solo penaliza volatilidad negativa)")
print("=" * 72)
print(f"  {'Ticker':<12} {'N_ops':>6}  {'N_down':>7}  {'Sortino Estrat':>15}  {'Sortino B&H':>12}  {'Gana':>6}")
print("-" * 72)
for _, r in df_sortino.iterrows():
    marca = "✅" if r['Gana_BH'] else "❌"
    print(f"  {r['Ticker']:<12} {int(r['N_ops']):>6}  "
          f"{int(r['N_downside']):>7}  "
          f"{r['Sortino_Estrat']:>15.4f}  "
          f"{r['Sortino_BH']:>12.4f}  "
          f"{marca:>6}")
print("-" * 72)
print(f"  {'MEDIA':<12} {df_sortino['N_ops'].mean():>6.0f}  "
      f"{'':>7}  "
      f"{df_sortino['Sortino_Estrat'].mean():>15.4f}  "
      f"{df_sortino['Sortino_BH'].mean():>12.4f}")
print("=" * 72)
print(f"\n  Bancos con mejor Sortino que B&H: {df_sortino['Gana_BH'].sum()}/{len(df_sortino)}")

# --- Gráfico comparativo Sharpe vs Sortino ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

x     = range(len(df_sortino))
width = 0.35

# Sortino
axes[0].bar([i - width/2 for i in x], df_sortino['Sortino_Estrat'],
            width=width, label='DCA Selectivo', color='steelblue', alpha=0.8)
axes[0].bar([i + width/2 for i in x], df_sortino['Sortino_BH'],
            width=width, label='Buy & Hold',    color='salmon',    alpha=0.8)
axes[0].axhline(0,   color='black',  linewidth=0.8)
axes[0].axhline(1,   color='green',  linestyle='--', linewidth=1.2, label='Sortino = 1')
axes[0].axhline(0.5, color='orange', linestyle='--', linewidth=1.2, label='Sortino = 0.5')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(df_sortino['Ticker'], rotation=45)
axes[0].set_title('Sortino Ratio', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Sortino Ratio')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Comparativa Sharpe vs Sortino de la estrategia
df_merged = df_sortino.merge(df_sharpe[['Ticker', 'Sharpe_Estrat']], on='Ticker')
axes[1].bar([i - width/2 for i in x], df_merged['Sharpe_Estrat'],
            width=width, label='Sharpe Estrategia', color='steelblue', alpha=0.8)
axes[1].bar([i + width/2 for i in x], df_merged['Sortino_Estrat'],
            width=width, label='Sortino Estrategia', color='teal',      alpha=0.8)
axes[1].axhline(1, color='green', linestyle='--', linewidth=1.2, label='= 1')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(df_merged['Ticker'], rotation=45)
axes[1].set_title('Sharpe vs Sortino — Estrategia', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Ratio')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Sortino Ratio — DCA Selectivo vs Buy & Hold',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()